# PhishGuard.AI: fine-tune XLM-RoBERTa / IndicBERT on Colab

1. **Runtime → Change runtime type → T4 GPU**, then run the cells in order.
2. On your PC, make the upload bundle first:
   ```
   cd backend
   python -m ml.make_colab_bundle
   ```
   This creates `backend/phishguard_colab_bundle.zip` (code + dataset).
3. At the end, download `transformer_model.zip` and unzip it into `backend/ml/models/` on your PC. Restart the backend and the transformer appears next to the other three models.

Same 80/20 split (`random_state=42`) as `train.py`, so the scores are directly comparable.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
!pip install -q "transformers>=4.46" "accelerate>=0.34" sentencepiece scikit-learn pandas joblib

In [ ]:
from google.colab import files
up = files.upload()  # choose phishguard_colab_bundle.zip
!unzip -o -q phishguard_colab_bundle.zip && ls ml

## Train
Pick one base model. `xlm-roberta-base` covers all seven languages. For IndicBERT use `ai4bharat/IndicBERTv2-MLM-only` (you may need to accept its terms on Hugging Face and run `huggingface-cli login`).

In [ ]:
BASE_MODEL = "xlm-roberta-base"   # or "ai4bharat/IndicBERTv2-MLM-only"
!python -m ml.train_transformer --model $BASE_MODEL --epochs 3 --batch 32 --lr 2e-5

## Compare with the classical models

In [ ]:
import json, pandas as pd
m = json.load(open("ml/models/metrics.json"))
keys = [k for k in ["logistic_regression", "naive_bayes", "svm", "on_device", "transformer"] if k in m]
pd.DataFrame({k: {x: round(m[k][x] * 100, 2) for x in ["accuracy", "precision", "recall", "f1"]} for k in keys}).T

In [ ]:
!cd ml/models && zip -q -r ../../transformer_model.zip transformer metrics.json
files.download("transformer_model.zip")